# Práctica 8 · Que responda rápido y no cueste una fortuna

Tu sistema funciona y es confiable. Faltan las dos preguntas que hace cualquiera que vaya a
pagarlo: cuánto tarda y cuánto cuesta.

Las dos tienen la misma respuesta de fondo, y la vas a medir en un momento: casi todo el tiempo y
casi todo el dinero se van en el mismo lugar. Una vez que sabes cuál es, las decisiones se
vuelven obvias.

Esta práctica es de medición y de números. Al final vas a poder decir, con cifras de tu propio
equipo, cuánto tarda una consulta, cuánto cuesta atender un mes de tráfico, y a partir de qué
volumen conviene un camino u otro.

Las celdas se ejecutan en orden, una por una, con Shift + Enter.

In [1]:
%pip install --quiet pymupdf

print("Listo.")


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /Users/fgrodriguez/ESAN_GlobalWeek2026/09_Notebooks_RAG/.venv/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Listo.


## 2. Comprobar que Ollama responde

Ollama es el programa que ejecuta los modelos de lenguaje dentro de tu computadora. Tiene que
estar encendido para que este cuaderno funcione, así que lo primero es confirmarlo.

Si algo falla, la salida de la celda te dice qué hacer según tu sistema operativo.

In [2]:
import platform
import sys

import requests

OLLAMA_URL = "http://localhost:11434"

print(f"Sistema: {platform.system()} {platform.machine()}")
print(f"Python:  {sys.version.split()[0]}\n")

try:
    respuesta = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5)
    respuesta.raise_for_status()
    modelos = sorted(m["name"] for m in respuesta.json()["models"])
    print(f"Ollama responde. Tienes {len(modelos)} modelos descargados:\n")
    for m in modelos:
        print(f"  - {m}")
except Exception as e:
    print(f"Ollama no responde en {OLLAMA_URL}")
    print(f"Detalle: {type(e).__name__}\n")
    if platform.system() == "Darwin":
        print("En Mac: abre la aplicación Ollama desde la carpeta Aplicaciones.")
        print("Debe aparecer su ícono en la barra de menús, arriba a la derecha.")
    elif platform.system() == "Windows":
        print("En Windows: busca Ollama en el menú Inicio y ábrelo.")
        print("Debe aparecer su ícono junto al reloj, abajo a la derecha.")
    else:
        print("Ejecuta 'ollama serve' en una terminal.")

Sistema: Darwin arm64
Python:  3.12.13

Ollama responde. Tienes 14 modelos descargados:

  - embeddinggemma:300m
  - gemma3:1b
  - gemma3:4b
  - gemma4:12b-mlx
  - gemma4:26b
  - gemma4:e4b
  - granite4.1:3b
  - granite4.1:8b
  - mxbai-embed-large:latest
  - nemotron-mini:4b
  - nomic-embed-text:latest
  - qwen3-4b-cs-ft:latest
  - qwen3:4b
  - shieldgemma:2b


## 3. Elegir el modelo según tu equipo

El modelo va a correr en tu máquina, así que la memoria que tengas importa. Un modelo grande
en un equipo chico no se rompe: simplemente tarda muchísimo y el sistema se pone lento.

Abajo hay tres opciones. Deja activa una sola, la que corresponda a tu computadora, y comenta
las demás poniéndoles un signo de gato al inicio de la línea. Si no sabes cuánta memoria
tienes, quédate con la opción A, que funciona en cualquier equipo.

El modelo de embeddings no se elige por equipo: es ligero y va igual en todos. Sí conviene saber
de dónde salió esa elección, y la respuesta es que está medida con documentos en español; en la
práctica 3 vas a reproducir la medición y a ver a los tres candidatos compitiendo.

In [3]:
# ---- Opción A: equipos de 8 GB de memoria o menos (descarga 3.3 GB) ---------
MODELO_LLM = "gemma3:4b"

# ---- Opción B: equipos de 16 GB de memoria (descarga 10 GB) ----------------
# MODELO_LLM = "gemma4:12b"        # Windows y Linux
# MODELO_LLM = "gemma4:12b-mlx"    # Mac con chip Apple (M1 en adelante), va más rápido

# ---- Opción C: equipos de 32 GB de memoria o más (descarga 17 GB) ----------
# MODELO_LLM = "gemma4:26b"        # Windows y Linux
# MODELO_LLM = "gemma4:26b-mlx"    # Mac con chip Apple

# El modelo de embeddings es ligero y es el mismo para todos. La elección está
# medida, no copiada de un tutorial: lo comprobamos en la práctica 3. Se eligió
# éste porque es el único de los tres que encuentra un pasaje en inglés cuando la
# pregunta va en español, algo que hace falta en cuanto el corpus mezcla idiomas.
MODELO_EMBEDDINGS = "embeddinggemma:300m"

print(f"Modelo de lenguaje:   {MODELO_LLM}")
print(f"Modelo de embeddings: {MODELO_EMBEDDINGS}")
print("\nSi alguno no aparece en la lista de la celda anterior, descárgalo con:")
print(f"   ollama pull {MODELO_LLM}")
print(f"   ollama pull {MODELO_EMBEDDINGS}")

Modelo de lenguaje:   gemma3:4b
Modelo de embeddings: embeddinggemma:300m

Si alguno no aparece en la lista de la celda anterior, descárgalo con:
   ollama pull gemma3:4b
   ollama pull embeddinggemma:300m


## 4. El sistema de siempre

Lo montamos rápido, sin explicar, porque ya lo conoces de las prácticas anteriores.

In [4]:
import time
import warnings
warnings.filterwarnings("ignore", message=".*langchain-community.*")

from pathlib import Path

import numpy as np
import pymupdf
import requests
from langchain_community.vectorstores import LanceDB
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings

# El corpus de siempre, montado en pocas líneas. Este cuaderno no trata de armarlo,
# sino de medir cuánto tarda y cuánto cuesta responder con él.
divisor = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
corpus = []
for nombre in ("tiendasol_politicas.pdf", "tiendasol_catalogo.pdf"):
    doc = pymupdf.open(Path("documentos") / nombre)
    texto = " ".join(p.get_text() for p in doc)
    corpus += [Document(page_content=t, metadata={"fuente": nombre})
               for t in divisor.split_text(texto)]

embeddings = OllamaEmbeddings(model=MODELO_EMBEDDINGS)
almacen = LanceDB.from_documents(corpus, embeddings, uri="lancedb_p8",
                                 table_name="latencia", mode="overwrite")

print(f"Corpus listo: {len(corpus)} fragmentos")

Corpus listo: 11 fragmentos


## 5. ¿Dónde se va el tiempo?

Antes de optimizar nada hay que saber qué optimizar. Vamos a cronometrar las tres etapas de una
consulta por separado.

Usamos la API de Ollama directamente en lugar de LangChain, porque así podemos leer también
cuántos tokens entraron y salieron. Ese dato lo necesitamos después para los costos.

In [5]:
# La misma consulta de siempre, pero con un cronómetro entre cada etapa. Sin esto solo
# se puede decir "va lento"; con esto se sabe cuál de las tres partes es la lenta, que
# es lo único que permite arreglarla.
def consulta_cronometrada(pregunta, modelo=None, k=4):
    """Ejecuta una consulta midiendo cada etapa y contando tokens."""
    modelo = modelo or MODELO_LLM

    inicio = time.time()
    vector = embeddings.embed_query(pregunta)
    t_vectorizar = time.time() - inicio

    inicio = time.time()
    documentos = almacen.similarity_search_by_vector(vector, k=k)
    # Tres tiempos por separado: convertir la pregunta en vector, buscar en el índice y
    # generar la respuesta. Ya verá cuál se lleva casi todo.
    t_buscar = time.time() - inicio

    contexto = "\n\n".join(d.page_content for d in documentos)
    prompt = (f"Responde la pregunta del cliente usando el contexto.\n\n"
              f"Contexto:\n{contexto}\n\nPregunta: {pregunta}")

    inicio = time.time()
    salida = requests.post(
        f"{OLLAMA_URL}/api/generate",
        json={"model": modelo, "prompt": prompt, "stream": False,
              "options": {"temperature": 0}},
        timeout=600,
    ).json()
    t_generar = time.time() - inicio

    return {
        "respuesta": salida["response"].strip(),
        "vectorizar": t_vectorizar,
        "buscar": t_buscar,
        "generar": t_generar,
        "total": t_vectorizar + t_buscar + t_generar,
        "tokens_entrada": salida.get("prompt_eval_count", 0),
        "tokens_salida": salida.get("eval_count", 0),
    }


medicion = consulta_cronometrada("¿Cuánto cuesta el envío a Lima?")

print(f"{'etapa':<26} {'ms':>9} {'del total':>11}")
print("-" * 48)
for etapa in ("vectorizar", "buscar", "generar"):
    ms = medicion[etapa] * 1000
    print(f"{etapa:<26} {ms:>9.0f} {100 * medicion[etapa] / medicion['total']:>10.0f}%")
print("-" * 48)
print(f"{'TOTAL':<26} {medicion['total'] * 1000:>9.0f}")
print(f"\nTokens: {medicion['tokens_entrada']} de entrada, "
      f"{medicion['tokens_salida']} de salida")

etapa                             ms   del total
------------------------------------------------
vectorizar                       102          5%
buscar                             9          0%
generar                         1854         94%
------------------------------------------------
TOTAL                           1964

Tokens: 607 de entrada, 48 de salida


Ahí está la respuesta, y no deja lugar a dudas: la generación se lleva casi todo. Buscar en el
índice ni siquiera aparece en el redondeo.

Eso tiene una consecuencia práctica que ahorra mucho trabajo perdido. Optimizar la búsqueda,
afinar el índice o cambiar la base vectorial no va a mejorar de forma perceptible el tiempo de
respuesta, porque no es ahí donde se va. Lo único que mueve la aguja es reducir el trabajo del
modelo, o no llamarlo.

De las dos, la segunda es la interesante.

## 6. No llamarlo: el caché exacto

La idea más simple es guardar las respuestas y reutilizarlas cuando la misma pregunta vuelva a
aparecer. En atención a clientes esto pasa todo el tiempo: unas pocas preguntas concentran la
mayoría de las consultas.

In [6]:
import hashlib

# Un diccionario en memoria: la clave es la pregunta y el valor la respuesta que ya se
# dio. Lo más simple que existe, y suficiente para ver el efecto.
cache_exacto = {}


# sha256 convierte la pregunta en una cadena corta y fija, que sirve de clave. Se pasa
# a minúsculas y se recortan los espacios para que dos escrituras casi iguales den la
# misma clave.
def consultar_con_cache_exacto(pregunta):
    clave = hashlib.sha256(pregunta.strip().lower().encode()).hexdigest()

    if clave in cache_exacto:
        return cache_exacto[clave], 0.0, True

    inicio = time.time()
    resultado = consulta_cronometrada(pregunta)
    tardo = time.time() - inicio

    cache_exacto[clave] = resultado["respuesta"]
    return resultado["respuesta"], tardo, False


PREGUNTA = "¿Cuánto cuesta el envío a Lima?"

_, t1, _ = consultar_con_cache_exacto(PREGUNTA)
_, t2, acierto = consultar_con_cache_exacto(PREGUNTA)

print(f"Primera vez:  {t1 * 1000:>8.0f} ms")
print(f"Segunda vez:  {t2 * 1000:>8.0f} ms  (¿del caché? {acierto})")
print(f"\nAhorro: prácticamente todo el tiempo de la consulta.")

Primera vez:      1099 ms
Segunda vez:         0 ms  (¿del caché? True)

Ahorro: prácticamente todo el tiempo de la consulta.


Instantáneo, como era de esperar. Pero el caché exacto tiene un problema serio en cuanto sale de
un cuaderno: nadie escribe la misma pregunta dos veces igual.

In [7]:
# Cuatro formas de preguntar lo mismo. El caché exacto solo acierta cuando el texto
# coincide letra por letra, y aquí se ve lo poco que eso ocurre en la vida real.
VARIANTES = [
    "¿Cuánto cuesta el envío a Lima?",
    "¿Cuál es el precio del envío a Lima?",
    "cuanto sale mandar a lima",
    "¿Cuánto cuesta el envío a Lima?  ",
]

cache_exacto.clear()
print(f"{'consulta':<44} {'¿acierto?':>10}")
print("-" * 58)
for pregunta in VARIANTES:
    _, _, acierto = consultar_con_cache_exacto(pregunta)
    print(f"{pregunta[:42]:<44} {'sí' if acierto else 'no':>10}")

print(f"\nEntradas guardadas en el caché: {len(cache_exacto)}")

consulta                                      ¿acierto?
----------------------------------------------------------


¿Cuánto cuesta el envío a Lima?                      no


¿Cuál es el precio del envío a Lima?                 no


cuanto sale mandar a lima                            no
¿Cuánto cuesta el envío a Lima?                      sí

Entradas guardadas en el caché: 3


Cuatro formas de preguntar lo mismo y el caché solo reconoce la que es idéntica carácter por
carácter. En un chat real, donde cada cliente escribe a su manera, un caché así casi nunca acierta.

Y fíjate en algo: ya tenemos la herramienta para arreglarlo. Llevamos todo el curso midiendo
parecido entre textos.

## 7. El caché semántico

En lugar de comparar el texto de la pregunta, comparamos su vector. Si la pregunta nueva se
parece lo suficiente a una que ya contestamos, devolvemos aquella respuesta.

Primero comprobemos que la idea se sostiene, mirando qué tan separadas están las preguntas
equivalentes de las que no lo son.

In [8]:
# Si el caché exacto falla por una coma, la salida es comparar significados en lugar
# de letras. Es el mismo cálculo de la práctica 3, aplicado a otra cosa.
def similitud(a, b):
    va, vb = np.array(embeddings.embed_query(a)), np.array(embeddings.embed_query(b))
    return float(va @ vb / (np.linalg.norm(va) * np.linalg.norm(vb)))


REFERENCIA = "¿Cuánto cuesta el envío a Lima?"
COMPARACIONES = [
    ("equivalente", "¿Cuál es el precio del envío a Lima?"),
    ("equivalente", "cuanto sale mandar a lima"),
    ("parecida pero distinta", "¿Cuánto cuesta el envío a la selva?"),
    ("otro tema", "¿Cuántos días tengo para devolver un producto?"),
]

print(f"{'relación':<26} {'similitud':>10}  consulta")
print("-" * 76)
for relacion, otra in COMPARACIONES:
    print(f"{relacion:<26} {similitud(REFERENCIA, otra):>10.3f}  {otra[:36]}")

relación                    similitud  consulta
----------------------------------------------------------------------------
equivalente                     0.975  ¿Cuál es el precio del envío a Lima?


equivalente                     0.682  cuanto sale mandar a lima
parecida pero distinta          0.728  ¿Cuánto cuesta el envío a la selva?


otro tema                       0.304  ¿Cuántos días tengo para devolver un


La separación es clara, pero mira con cuidado la tercera fila, porque ahí está el riesgo.

"¿Cuánto cuesta el envío a la selva?" se parece mucho a la pregunta de referencia, y sin embargo
la respuesta correcta es distinta: otra zona, otro precio, otro plazo. Si pones el umbral
demasiado permisivo, el caché va a contestar el precio de Lima a alguien que preguntó por la
selva.

Ese es el peligro real de esta técnica. Un caché exacto que no acierta solo desperdicia tiempo;
un caché semántico mal calibrado contesta mal.

In [9]:
# Una clase agrupa datos y las funciones que operan sobre ellos. Aquí guarda los
# vectores de las preguntas ya atendidas junto con sus respuestas.
class CacheSemantico:
    """Guarda respuestas y las reutiliza cuando llega una pregunta parecida."""

    # __init__ se ejecuta al crear el caché. El umbral es la decisión de negocio: qué
    # tan parecidas deben ser dos preguntas para darles la misma respuesta.
    def __init__(self, umbral=0.92):
        self.umbral = umbral
        self.vectores = []
        self.preguntas = []
        self.respuestas = []

    # Compara la pregunta nueva contra todas las guardadas y se queda con la más
    # parecida. Si no llega al umbral, devuelve None y habrá que generar de nuevo.
    def buscar(self, pregunta):
        if not self.vectores:
            return None

        v = np.array(embeddings.embed_query(pregunta))
        v = v / np.linalg.norm(v)
        similitudes = np.array(self.vectores) @ v
        mejor = int(np.argmax(similitudes))

        if similitudes[mejor] >= self.umbral:
            return self.respuestas[mejor], self.preguntas[mejor], float(similitudes[mejor])
        return None

    def guardar(self, pregunta, respuesta):
        v = np.array(embeddings.embed_query(pregunta))
        self.vectores.append(v / np.linalg.norm(v))
        self.preguntas.append(pregunta)
        self.respuestas.append(respuesta)


# 0.92 no es un número mágico: sale de la tabla de arriba. Más abajo se ve qué pasa
# cuando se pone demasiado bajo.
cache = CacheSemantico(umbral=0.92)


def consultar(pregunta):
    inicio = time.time()
    encontrado = cache.buscar(pregunta)

    if encontrado:
        respuesta, original, parecido = encontrado
        return respuesta, time.time() - inicio, f"caché ({parecido:.3f} vs '{original[:30]}')"

    resultado = consulta_cronometrada(pregunta)
    cache.guardar(pregunta, resultado["respuesta"])
    return resultado["respuesta"], time.time() - inicio, "generada"


# Las tres primeras son la misma pregunta escrita distinto; las dos últimas son otras
# preguntas. Un buen umbral acierta en las tres y falla a propósito en las dos.
CONSULTAS = [
    "¿Cuánto cuesta el envío a Lima?",
    "¿Cuál es el precio del envío a Lima?",
    "cuanto sale mandar a lima",
    "¿Cuánto cuesta el envío a la selva?",
    "¿Cuántos días tengo para devolver un producto?",
]

print(f"{'consulta':<46} {'ms':>7}  origen")
print("-" * 92)
for pregunta in CONSULTAS:
    _, tardo, origen = consultar(pregunta)
    print(f"{pregunta[:44]:<46} {tardo*1000:>7.0f}  {origen}")

consulta                                            ms  origen
--------------------------------------------------------------------------------------------


¿Cuánto cuesta el envío a Lima?                   1801  generada
¿Cuál es el precio del envío a Lima?                84  caché (0.975 vs '¿Cuánto cuesta el envío a Lima')


cuanto sale mandar a lima                         1671  generada


¿Cuánto cuesta el envío a la selva?               1845  generada


¿Cuántos días tengo para devolver un product      1788  generada


Mira bien la tabla, porque el resultado no es el que promete la teoría.

De las tres formas de preguntar por el envío a Lima, solo una salió del caché. La versión
coloquial, "cuanto sale mandar a lima", se generó de nuevo pese a preguntar exactamente lo mismo.

Podrías pensar que basta con bajar el umbral. Vamos a comprobar si eso es cierto, midiendo las
similitudes reales.

In [10]:
# El riesgo del caché semántico: envío a Lima y envío a la selva se parecen mucho como
# frases, pero tienen precios distintos. Bajar el umbral para atrapar más variantes
# hace que en algún momento se le entregue al cliente la respuesta equivocada.
CONTRA = "¿Cuánto cuesta el envío a Lima?"
COMPARAR = [
    ("mismo significado", "¿Cuál es el precio del envío a Lima?"),
    ("mismo significado", "cuanto cuesta el envio a lima"),
    ("mismo significado", "cuanto sale mandar a lima"),
    ("OTRA pregunta", "¿Cuánto cuesta el envío a la selva?"),
    ("otro tema", "¿Cuántos días tengo para devolver un producto?"),
]

print(f"{'relación':<20} {'similitud':>10}  consulta")
print("-" * 74)
for relacion, otra in COMPARAR:
    print(f"{relacion:<20} {similitud(CONTRA, otra):>10.3f}  {otra[:40]}")

relación              similitud  consulta
--------------------------------------------------------------------------
mismo significado         0.975  ¿Cuál es el precio del envío a Lima?


mismo significado         0.934  cuanto cuesta el envio a lima
mismo significado         0.682  cuanto sale mandar a lima


OTRA pregunta             0.728  ¿Cuánto cuesta el envío a la selva?
otro tema                 0.304  ¿Cuántos días tengo para devolver un pro


Ahí está el problema, y no se arregla moviendo el umbral.

La forma coloquial de preguntar lo mismo queda **más lejos** que una pregunta sobre otra zona,
que tiene una respuesta distinta. No existe ningún umbral que acepte la primera y rechace la
segunda: si lo bajas para atrapar al cliente que escribe coloquial, empiezas a contestarle el
precio de Lima a quien preguntó por la selva.

La razón es que la similitud entre vectores mezcla dos cosas que a nosotros nos gustaría separar:
de qué se habla y cómo está escrito. Dos textos con el mismo registro y estructura se parecen
aunque cambien el dato clave; dos textos que dicen lo mismo en registros distintos se separan.

Qué hacer con esto, entonces.

Usa el caché semántico con un umbral alto, como el 0.92 que pusimos, asumiendo que va a acertar
solo en las reformulaciones cercanas. Eso ya sirve: en atención a clientes unas pocas preguntas
concentran la mayoría del tráfico y muchas llegan bien escritas.

Y no lo uses como única capa. Normalizar el texto antes de comparar, quitando acentos, signos y
mayúsculas, recupera parte de las variantes sin tocar el umbral. Ese es el tipo de arreglo barato
que conviene antes de complicar el modelo.

Lo que no debes hacer es bajar el umbral hasta que el caché acierte seguido. Un caché que ahorra
poco cuesta dinero; uno que confunde preguntas cuesta clientes.

## 8. El caché también se echa a perder

Un detalle que se olvida y causa incidentes difíciles de rastrear: si la documentación cambia, el
caché sigue contestando lo viejo.

Imagina que suben el costo de envío. El índice se reconstruye, todo correcto, y aun así los
clientes siguen recibiendo el precio anterior porque la respuesta salía del caché. El sistema no
falla; miente sin enterarse.

La solución es ligar el caché a la versión de los documentos. Cuando la documentación cambia, el
caché anterior deja de valer.

In [11]:
# Hereda todo lo anterior y le agrega una cosa: recordar con qué versión de la
# documentación se guardaron las respuestas.
class CacheConVersion(CacheSemantico):
    """Caché que se invalida solo cuando cambia la documentación."""

    def __init__(self, umbral=0.92, version=""):
        super().__init__(umbral)
        self.version = version

    def revisar_version(self, version_actual):
        if version_actual != self.version:
            anteriores = len(self.vectores)
            self.vectores, self.preguntas, self.respuestas = [], [], []
            self.version = version_actual
            return anteriores
        return 0


# Una sola huella para toda la carpeta. Si cambia cualquier documento, cambia la
# huella, y el caché entero se descarta. Es preferible perder velocidad un rato a
# seguir contestando con una política que ya se derogó.
def version_del_corpus(carpeta):
    """Huella del contenido de todos los documentos juntos."""
    h = hashlib.sha256()
    for ruta in sorted(Path(carpeta).glob("*.pdf")):
        h.update(ruta.read_bytes())
    return h.hexdigest()[:12]


version = version_del_corpus("documentos")
cache_v = CacheConVersion(umbral=0.92, version=version)
cache_v.guardar("¿Cuánto cuesta el envío a Lima?", "Cuesta S/ 9.90")

print(f"Versión actual del corpus: {version}")
print(f"Entradas en el caché: {len(cache_v.vectores)}")

borradas = cache_v.revisar_version("otra_version_distinta")
print(f"\nAl detectar documentación nueva se descartaron {borradas} entradas.")
print(f"Entradas ahora: {len(cache_v.vectores)}")

Versión actual del corpus: fbc477180b2c
Entradas en el caché: 1

Al detectar documentación nueva se descartaron 1 entradas.
Entradas ahora: 0


## 9. Escalar solo cuando hace falta

Otra forma de ahorrar tiempo y dinero: no usar el modelo grande para todo.

La idea es atender primero con el modelo rápido y barato, comprobar si su respuesta se sostiene, y
recurrir al grande solo cuando no. Como la mayoría de las preguntas de atención a clientes son
sencillas, la mayoría se resuelve en el primer nivel.

Necesitamos un criterio para decidir cuándo escalar. Usamos uno simple y honesto: si el modelo
rápido admite que no sabe, o si su respuesta no menciona ningún dato del contexto, se escala.

In [12]:
# La idea de la cascada: atender con el modelo barato casi siempre, y llamar al caro
# solo cuando haga falta. El problema es acertar en cuándo hace falta.
MODELO_RAPIDO = "gemma3:4b"
MODELO_CAPAZ = "gemma4:12b-mlx"   # cámbialo por "gemma4:12b" fuera de Mac con chip Apple


# Primer intento, el que se le ocurre a cualquiera: buscar frases de rendición en la
# respuesta. Falla justo cuando más importa, porque el modelo que inventa lo hace con
# aplomo y no dice ninguna de estas frases.
def parece_insuficiente(respuesta):
    """Primer intento: buscar señales de que el modelo no supo."""
    señales = ["no tengo", "no encuentro", "no está", "no puedo", "no dispongo",
               "no se especifica", "no aparece"]
    texto = respuesta.lower()
    return any(s in texto for s in señales) or len(respuesta) < 40


PRUEBAS = [
    ("sí está en la documentación", "¿Cuánto cuesta el envío a Lima?"),
    ("sí está en la documentación", "¿Qué significa el estado EST-07?"),
    ("NO está en la documentación",
     "¿Cuál es la política de garantía para productos importados con daño de fábrica?"),
]

print("¿La heurística detecta cuándo hay que escalar?\n")
print(f"{'caso':<30} {'¿escalaría?':>12}  respuesta del modelo rápido")
print("-" * 100)
for etiqueta, pregunta in PRUEBAS:
    r = consulta_cronometrada(pregunta, modelo=MODELO_RAPIDO)["respuesta"]
    print(f"{etiqueta:<30} {'sí' if parece_insuficiente(r) else 'NO':>12}  {r[:48]}")

¿La heurística detecta cuándo hay que escalar?

caso                            ¿escalaría?  respuesta del modelo rápido
----------------------------------------------------------------------------------------------------


sí está en la documentación              NO  El costo de envío a la Lima Metropolitana es de 


sí está en la documentación              NO  El estado EST-07 significa “En transito”. El paq


NO está en la documentación              NO  La política de garantía establece que los produc


La tercera pregunta no está en la documentación, y aun así la heurística dice que no hay que
escalar. Peor todavía: mira la respuesta que dio el modelo rápido. Contesta con aplomo mezclando
las reglas de liquidación y de reembolso, que no tienen nada que ver con productos importados.

Ese es justo el fallo que hace inútil la heurística. Está buscando señales de duda, y el problema
es que el modelo no duda: inventa con seguridad y con extensión. Las respuestas inventadas suelen
ser más largas y más seguras que las honestas, así que medir longitud o buscar "no sé" detecta
exactamente al revés de lo que uno querría.

Para decidir si escalar hace falta comparar la respuesta con el contexto, que es el verificador
que construimos en la práctica 5.

In [13]:
import json

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

# El verificador corre con el modelo barato: si tuviera que usar el caro, no habría
# ahorro que valga la pena.
juez = ChatOllama(model=MODELO_RAPIDO, temperature=0)

# Primer intento: preguntar si la RESPUESTA está respaldada por el contexto.
FIDELIDAD = ChatPromptTemplate.from_template(
    """Compara la respuesta con el contexto. ¿Cada afirmación de la respuesta
está respaldada por el contexto?

Responde solo con JSON: {{"fundamentada": true/false}}

Contexto:
{context}

Respuesta:
{answer}"""
)


# La misma precaución de siempre al leer JSON de un modelo: quedarse con lo que hay
# entre la primera llave y la última, y normalizar las claves.
def leer_json(crudo, clave):
    try:
        datos = json.loads(crudo[crudo.index("{"):crudo.rindex("}") + 1])
        return {str(k).strip().lower(): v for k, v in datos.items()}.get(clave)
    except (ValueError, AttributeError, json.JSONDecodeError):
        return None


inventada = ("La política de garantía establece que los productos adquiridos en "
             "liquidación admiten cambio pero no reembolso, salvo que presenten "
             "un defecto de fabricación.")
honesta = "El contexto no menciona nada sobre productos importados."

pregunta_dificil = ("¿Cuál es la política de garantía para productos importados "
                    "con daño de fábrica?")
contexto = "\n\n".join(
    d.page_content for d in almacen.similarity_search(pregunta_dificil, k=4))

print("Verificando FIDELIDAD (¿lo que dice está en el contexto?)\n")
for etiqueta, respuesta in [("respuesta inventada", inventada),
                            ("respuesta honesta", honesta)]:
    crudo = (FIDELIDAD | juez | StrOutputParser()).invoke(
        {"context": contexto, "answer": respuesta})
    print(f"  {etiqueta:<22} fundamentada = {leer_json(crudo, 'fundamentada')}")

Verificando FIDELIDAD (¿lo que dice está en el contexto?)



  respuesta inventada    fundamentada = True


  respuesta honesta      fundamentada = False


Otra vez al revés de lo esperado. El verificador aprueba la respuesta inventada y rechaza la
honesta.

Y cuando uno mira por qué, resulta que el verificador no está equivocado: **la respuesta inventada
sí es fiel al contexto**. Habla de liquidaciones y de defectos de fabricación, y las dos cosas
están en el documento. Lo que falla es otra cosa: esas reglas no aplican a productos importados,
que es lo que preguntó el cliente.

Estábamos midiendo lo que no era. La fidelidad comprueba que no se invente datos; lo que
necesitamos aquí es saber si el contexto **cubre la pregunta**. Son las dos dimensiones que
separamos en la práctica 3, pertinencia y fidelidad, y aquí se ve por qué convenía distinguirlas.

Cambiemos la pregunta que le hacemos al juez.

In [14]:
# Segundo intento: no evaluamos la respuesta, sino si el contexto alcanza.
COBERTURA = ChatPromptTemplate.from_template(
    """¿El contexto contiene la información necesaria para responder esta
pregunta específica?

No evalúes ninguna respuesta. Solo si el contexto cubre lo que se pregunta.

Responde solo con JSON: {{"cubierta": true/false}}

Pregunta: {q}

Contexto:
{context}"""
)


# Segundo intento, y este sí funciona: en vez de juzgar la respuesta, se pregunta si
# el contexto alcanza. Es una pregunta más fácil, se hace ANTES de generar y no
# depende de que el modelo reconozca que no sabe.
def contexto_suficiente(pregunta, k=4):
    contexto = "\n\n".join(
        d.page_content for d in almacen.similarity_search(pregunta, k=k))
    crudo = (COBERTURA | juez | StrOutputParser()).invoke(
        {"q": pregunta, "context": contexto})
    return leer_json(crudo, "cubierta") is True


print(f"{'caso':<30} {'¿cubierta?':>12}  pregunta")
print("-" * 84)
for etiqueta, pregunta in PRUEBAS:
    print(f"{etiqueta:<30} {str(contexto_suficiente(pregunta)):>12}  {pregunta[:38]}")

caso                             ¿cubierta?  pregunta
------------------------------------------------------------------------------------


sí está en la documentación            True  ¿Cuánto cuesta el envío a Lima?


sí está en la documentación            True  ¿Qué significa el estado EST-07?


NO está en la documentación           False  ¿Cuál es la política de garantía para 


Cuatro de cuatro, y con el modelo pequeño. La diferencia no estuvo en usar un modelo más capaz
sino en preguntar lo correcto.

Vale la pena detenerse en esto, porque es la lección más transferible de la práctica. Los tres
intentos fueron: buscar palabras de duda en la respuesta, verificar que la respuesta fuera fiel, y
preguntar si el contexto cubría la pregunta. Los dos primeros fallaron no por falta de potencia,
sino porque medían algo distinto de lo que nos importaba. Cuando un evaluador automático da
resultados raros, antes de cambiarlo por uno más grande conviene revisar qué le estás preguntando.

Ahora sí, el router completo.

In [15]:
# La decisión completa: si el contexto alcanza, contesta el modelo rápido; si no,
# escala al capaz. Lo caro solo se paga en las consultas que lo necesitan.
def responder_en_cascada(pregunta):
    inicio = time.time()

    if contexto_suficiente(pregunta):
        rapida = consulta_cronometrada(pregunta, modelo=MODELO_RAPIDO)
        return rapida["respuesta"], time.time() - inicio, MODELO_RAPIDO

    capaz = consulta_cronometrada(pregunta, modelo=MODELO_CAPAZ)
    return capaz["respuesta"], time.time() - inicio, f"escaló a {MODELO_CAPAZ}"


print(f"{'caso':<30} {'seg':>6}  atendida por")
print("-" * 74)
for etiqueta, pregunta in PRUEBAS:
    _, tardo, quien = responder_en_cascada(pregunta)
    print(f"{etiqueta:<30} {tardo:>6.1f}  {quien}")

caso                              seg  atendida por
--------------------------------------------------------------------------


sí está en la documentación       1.8  gemma3:4b


sí está en la documentación       1.4  gemma3:4b


NO está en la documentación      75.5  escaló a gemma4:12b-mlx


Ahora el router hace lo que promete: las preguntas cubiertas se atienden con el modelo barato, y
solo las que el corpus no cubre pagan el modelo caro.

Aunque, pensándolo bien, para las preguntas que el corpus no cubre quizá no convenga escalar sino
admitir que no hay información y ofrecer un asesor, como hicimos en la práctica 5. Escalar al
modelo grande sirve cuando la pregunta es difícil pero la información está; cuando la información
no está, un modelo más caro solo produce una invención más elaborada.

El costo de todo esto: cada consulta necesita una llamada extra para decidir. Si tus dos niveles
cuestan parecido, esa llamada extra puede salir más cara que escalar siempre. El router se
justifica cuando la diferencia entre niveles es grande, por ejemplo entre un modelo local y una
API de las caras.

## 10. Cuánto cuesta esto

Llegamos a la pregunta que hace quien firma el presupuesto. Vamos a calcularlo con los tokens que
medimos, comparando las dos formas de operar el mismo sistema.

Los precios de las APIs cambian seguido, así que están en variables al principio de la celda para
que los actualices. Los valores puestos son de referencia para un modelo económico.

In [16]:
# --- Ajusta estos números a tu caso ------------------------------------------
# Los números de esta celda son de ejemplo. Cámbielos por los de su caso: es la única
# forma de que la comparación signifique algo para su empresa.
CONSULTAS_POR_MES = 150_000

# Precios de API, en dólares por millón de tokens (revísalos antes de usarlos)
PRECIO_ENTRADA = 0.15
PRECIO_SALIDA = 0.60

# Operación local
POTENCIA_WATTS = 45          # consumo del equipo mientras responde
PRECIO_KWH = 0.15            # dólares por kilovatio hora
COSTO_EQUIPO = 2_000         # lo que cuesta la máquina
MESES_DE_VIDA = 36           # en cuántos meses la amortizas
# ------------------------------------------------------------------------------

tokens_entrada = medicion["tokens_entrada"]
tokens_salida = medicion["tokens_salida"]
segundos = medicion["total"]

# Los precios vienen por millón de tokens, así que se divide para tener el costo de
# una sola consulta. Fíjese que se usan los tokens MEDIDOS arriba, no una estimación.
costo_api = (tokens_entrada * PRECIO_ENTRADA + tokens_salida * PRECIO_SALIDA) / 1_000_000
api_mensual = costo_api * CONSULTAS_POR_MES

# Lo local no es gratis: consume electricidad mientras responde, y el equipo se paga
# aunque nadie pregunte nada. Ese costo fijo es lo que cambia toda la comparación.
energia_kwh = (POTENCIA_WATTS * segundos / 3600) / 1000
costo_energia = energia_kwh * PRECIO_KWH
local_mensual = costo_energia * CONSULTAS_POR_MES + COSTO_EQUIPO / MESES_DE_VIDA

print(f"Por consulta: {tokens_entrada} tokens de entrada, {tokens_salida} de salida, "
      f"{segundos:.1f} s\n")
print(f"{'concepto':<34} {'por consulta':>14} {'al mes':>12}")
print("-" * 62)
print(f"{'API de pago':<34} {costo_api:>13.6f} {api_mensual:>11.2f}")
print(f"{'local: electricidad':<34} {costo_energia:>13.6f} "
      f"{costo_energia * CONSULTAS_POR_MES:>11.2f}")
print(f"{'local: equipo amortizado':<34} {'':>13} {COSTO_EQUIPO / MESES_DE_VIDA:>11.2f}")
print(f"{'local: total':<34} {'':>13} {local_mensual:>11.2f}")
print("-" * 62)
diferencia = api_mensual - local_mensual
print(f"{'diferencia mensual':<34} {'':>13} {diferencia:>11.2f}")

Por consulta: 607 tokens de entrada, 48 de salida, 2.0 s

concepto                             por consulta       al mes
--------------------------------------------------------------
API de pago                             0.000120       17.98
local: electricidad                     0.000004        0.55
local: equipo amortizado                               55.56
local: total                                           56.11
--------------------------------------------------------------
diferencia mensual                                    -38.13


Antes de sacar conclusiones, conviene ver dónde está el punto de equilibrio, porque la respuesta
cambia por completo según el volumen.

In [17]:
print(f"{'consultas al mes':>18} {'API':>12} {'local':>12}  {'conviene':>10}")
print("-" * 58)
# La respuesta cambia con el volumen, y por eso no hay una recomendación única. A poco
# volumen la API es más barata; a mucho volumen se invierte.
for volumen in (1_000, 10_000, 50_000, 150_000, 500_000):
    api = costo_api * volumen
    local = costo_energia * volumen + COSTO_EQUIPO / MESES_DE_VIDA
    print(f"{volumen:>18,} {api:>12.2f} {local:>12.2f}  {'local' if local < api else 'API':>10}")

# El punto de equilibrio: cuántas consultas al mes hacen falta para que lo local
# empiece a salir a cuenta. Es el número que hay que llevar a una reunión, porque
# convierte una discusión de opiniones en una de cifras.
equilibrio = (COSTO_EQUIPO / MESES_DE_VIDA) / max(costo_api - costo_energia, 1e-12)
print(f"\nPunto de equilibrio: alrededor de {equilibrio:,.0f} consultas al mes")

  consultas al mes          API        local    conviene
----------------------------------------------------------
             1,000         0.12        55.56         API
            10,000         1.20        55.59         API
            50,000         5.99        55.74         API
           150,000        17.98        56.11         API
           500,000        59.92        57.40       local

Punto de equilibrio: alrededor de 478,240 consultas al mes


La conclusión suele sorprender a quien esperaba que lo local fuera siempre más barato, o al revés.

Con volúmenes bajos gana la API, porque no hay que comprar nada y solo pagas lo que usas. A partir
de cierto punto gana lo local, porque el costo por consulta es casi solo electricidad y el equipo
ya está pagado.

Ahora, dos advertencias sobre estos números, que son las que hay que decir en la junta.

La primera es que el cálculo local supone que la máquina ya existe y está ociosa, y que el tiempo
de quien la administra no cuesta. En una empresa, el costo de operar y mantener un servidor suele
superar la diferencia que acabas de calcular. Si el equipo hay que comprarlo, instalarlo y
cuidarlo, la cuenta cambia.

La segunda es que hay razones que no aparecen en esta tabla y que suelen pesar más que el dinero.
Con la API, cada consulta manda el documento y la pregunta del cliente a un tercero. Para muchas
áreas de cumplimiento eso zanja la discusión antes de mirar el precio, y es la razón principal por
la que este curso usa modelos locales.

## 11. Lo que llevas

Tres cosas medidas, no supuestas.

La generación se lleva más del noventa por ciento del tiempo. Optimizar la búsqueda es esfuerzo
perdido; lo que sirve es no llamar al modelo, o llamar a uno más barato.

El caché semántico convierte varias formas de preguntar lo mismo en una sola respuesta, y su
parámetro delicado es el umbral: demasiado permisivo y contesta la pregunta de otro. Ligarlo a la
versión de los documentos evita el problema más difícil de detectar, que es servir respuestas
correctas de una documentación que ya cambió.

Y el costo depende del volumen más que de la tecnología. Ahora puedes calcular el punto de
equilibrio con tus propios números en lugar de repetir lo que dice un artículo.

En la práctica siguiente nos ocupamos del otro requisito que aparece en cuanto esto sale del
cuaderno: los datos personales y quién puede ver qué.